# Step 03 — Feature Engineering, Vectorization & Classical Modeling

## Objectif de ce notebook

Dans les étapes précédentes, nous avons construit un dataset nettoyé à partir de l’EDA :

- **Step 01** : compréhension du dataset et identification des anomalies.


L’objectif est de comparer des approches classiques et solides :

1. séparation Train / Validation / Test ;
2. pretratement sur text
3. encodage des labels ;
5. vectorisation TF-IDF ;
6. analyse des matrices TF-IDF ;
7. balance du dataset ;
8. entraînement de plusieurs modèles ;
9. vectorisation Word2Vec ;
10. comparaison TF-IDF vs Word2Vec ;
11. sélection du meilleur modèle à déployer.



## 0. Installation optionnelle des librairies

Exécuter ces installations seulement si certaines librairies ne sont pas encore installées.

In [1]:
!pip install gensim
!pip install xgboost
!pip install joblib

## 1. Initialisation

Cette cellule regroupe toutes les librairies nécessaires pour l’étape 03.

On utilise principalement :
- `scikit-learn` pour la vectorisation TF-IDF, le split, les modèles classiques et les métriques ;
- `gensim` pour Word2Vec ;
- `joblib` pour sauvegarder le meilleur modèle.

In [1]:
import os
import re
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.base import BaseEstimator, TransformerMixin

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

# 2. Chargement du dataset nettoyé


```text
mental_health_cleaned.csv
```

Il doit contenir au minimum :
- une colonne texte nettoyé : `clean_text`
- une colonne label : `status`



In [2]:
DATA_PATH = "data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset chargé avec succès.")
print("Dimensions :", df.shape)
display(df.head())
print("\nColonnes disponibles :")
print(df.columns.tolist())

Dataset chargé avec succès.
Dimensions : (53043, 3)


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety



Colonnes disponibles :
['Unnamed: 0', 'statement', 'status']


In [4]:
#supression des valeur manquantes
df = df.dropna()

#delete duplicates
df = df.drop_duplicates()

#supression des statement < 3
df = df[df["statement"].str.len() > 3]




## 4. Séparation X / y

On sépare :
- `X` : les textes nettoyés ;
- `y` : les classes de santé mentale.

Cette séparation permet de préparer la phase d’apprentissage.

In [17]:
TEXT_COL = "statement"
LABEL_COL = "status"

X = df[TEXT_COL]
y = df[LABEL_COL]

print("Nombre de textes :", len(X))
print("Nombre de labels :", len(np.unique(y)))
print("Exemple de texte nettoyé :")
print(X[0])
print("\nLabel associé :", y[0])

Nombre de textes : 52667
Nombre de labels : 7
Exemple de texte nettoyé :
oh my gosh

Label associé : Anxiety


In [6]:
## pretraitment

In [7]:
!pip install emoji contractions ftfy

In [18]:
import emoji
import contractions
import re
import ftfy

#convesion imoji
def convert_emojis(text):
    if emoji is not None:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace(":", " ").replace("_", " ")
    return text


def fix_unicode_text(text):
    return ftfy.fix_text(text)


def convert_emojis(text):
    if emoji is not None:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace(":", " ").replace("_", " ")
    return text

#slang words
slang_dict = {
    "u": "you",
    "ur": "your",
    "ya": "you",
    "idk": "i do not know",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "btw": "by the way",
    "brb": "be right back",
    "lol": "laughing",
    "lmao": "laughing",
    "omg": "oh my god",
    "tf": "the fuck",
    "wtf": "what the fuck",
    "rn": "right now",
    "tho": "though",
    "thx": "thanks",
    "pls": "please",
    "plz": "please",
    "bc": "because",
    "bcz": "because",
    "cuz": "because",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to",
    "kinda": "kind of",
    "sorta": "sort of"
}
def normalize_slang_text(text):
    words = text.split()
    normalized = [slang_dict.get(word, word) for word in words]
    return " ".join(normalized)



def reduce_repeated_punctuation(text):
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    return text


def remove_special_characters(text):
    # On garde lettres, chiffres, espaces, apostrophes, ! et ? car ils peuvent signaler une intensité émotionnelle.
    text = re.sub(r"[^a-zA-Z0-9\s'!?]", " ", text)
    return text


def preprocess_text(text):
    text = str(text)  # Ensure the input is a string
    text = text.lower()

    # Fix unicode
    text = ftfy.fix_text(text)

    # Expand contractions
    text = contractions.fix(text)

    # URLs
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)

    # Mentions
    text = re.sub(r"@\w+", " USER ", text)

    # Hashtags
    text = re.sub(r"#(\w+)", r"\1", text)

    # Emojis
    text = convert_emojis(text)

    # Slang
    text = normalize_slang_text(text)

    # Keep emotional punctuation
    text = re.sub(r"!{4,}", "!!!", text)
    text = re.sub(r"\?{4,}", "???", text)

    # Remove weird symbols ONLY
    text = re.sub(r"[^a-zA-Z0-9\s!?']", " ", text)

    # Clean spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


X = X.apply(preprocess_text)

In [ ]:
!python -m spacy download en_core_web_sm

In [20]:
#import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

stemmer = PorterStemmer()

def stem_text(text):
    tokens = word_tokenize(text)

    stemmed_tokens = [
        stemmer.stem(token)
        for token in tokens
    ]

    return " ".join(stemmed_tokens)

X = X.apply(stem_text)

[nltk_data] Downloading package punkt to /home/mohamed/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
X.isna().sum()

## 5. Train / Validation / Test Split

On crée trois ensembles :

- **Train set** : apprentissage des modèles ;
- **Validation set** : comparaison et réglage des modèles ;
- **Test set** : évaluation finale seulement à la fin.

On utilise `stratify=y` pour conserver la même proportion de classes dans chaque ensemble, ce qui est indispensable car le dataset est déséquilibré.

In [21]:
# 70% train, 15% validation, 15% test

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train :", len(X_train))
print("Validation :", len(X_val))
print("Test :", len(X_test))

split_summary = pd.DataFrame({
    "train": pd.Series(y_train).value_counts(normalize=True).sort_index(),
    "validation": pd.Series(y_val).value_counts(normalize=True).sort_index(),
    "test": pd.Series(y_test).value_counts(normalize=True).sort_index()
}).round(4) * 100

display(split_summary)

Train : 36866
Validation : 7900
Test : 7901


,train,validation,test
status,,,
Anxiety,7.29,7.29,7.30
Bipolar,5.27,5.28,5.27
Depression,29.25,29.24,29.25
Normal,31.01,31.00,31.01
Personality disorder,2.05,2.05,2.04
Stress,4.91,4.91,4.91
Suicidal,20.22,20.23,20.23


### Interprétation

Les proportions doivent être proches entre train, validation et test.

Si une classe minoritaire devient trop faible dans le test set, l’évaluation devient instable.  
C’est pour cela que le split stratifié est indispensable dans ce projet.

## 6. Encodage des labels

Les modèles ML travaillent avec des labels numériques.  
On transforme donc les classes textuelles en valeurs numériques avec `LabelEncoder`.

On garde aussi le mapping pour interpréter les résultats.

In [22]:
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

label_mapping = pd.DataFrame({
    "classe": label_encoder.classes_,
    "encoded_label": range(len(label_encoder.classes_))
})

display(label_mapping)

,classe,encoded_label
0,Anxiety,0
1,Bipolar,1
2,Depression,2
3,Normal,3
4,Personality disorder,4
5,Stress,5
6,Suicidal,6


In [23]:
tfidf_probe = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True,
    token_pattern=r"\b[a-zA-Z']{2,}\b"
)

X_train_tfidf = tfidf_probe.fit_transform(X_train)

print("Shape TF-IDF train :", X_train_tfidf.shape)
print("Taille du vocabulaire :", len(tfidf_probe.get_feature_names_out()))

non_zero = X_train_tfidf.nnz
total_values = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]
density = non_zero / total_values
sparsity = 1 - density

print("Densité :", round(density * 100, 4), "%")
print("Sparsity :", round(sparsity * 100, 4), "%")

print("\nExemples de features :")
print(tfidf_probe.get_feature_names_out()[:50])

Shape TF-IDF train : (36866, 10000)
Taille du vocabulaire : 10000
Densité : 1.0465 %
Sparsity : 98.9535 %

Exemples de features :
['abandon' 'abandon me' 'abdomen' 'abil' 'abil to' 'abilifi' 'abl'
 'abl to' 'abnorm' 'about' 'about all' 'about an' 'about and'
 'about anyth' 'about be' 'about but' 'about everyth' 'about get'
 'about go' 'about have' 'about her' 'about him' 'about how' 'about is'
 'about it' 'about kill' 'about life' 'about me' 'about minut'
 'about month' 'about my' 'about myself' 'about not' 'about other'
 'about peopl' 'about someth' 'about suicid' 'about that' 'about the'
 'about their' 'about them' 'about these' 'about thi' 'about thing'
 'about to' 'about two' 'about week' 'about what' 'about year' 'about you']


## 10. Fonction d’évaluation commune

On crée une fonction unique pour évaluer tous les modèles de la même manière.

On calcule :
- Accuracy ;
- Precision macro ;
- Recall macro ;
- F1 macro ;
- F1 weighted.

La métrique principale recommandée ici est **Macro F1**, car elle donne de l’importance à toutes les classes, même les minoritaires.

In [25]:
def evaluate_model(model_name, vectorizer_name, model, X_eval, y_eval_enc, label_encoder):
    y_pred = model.predict(X_eval)

    accuracy = accuracy_score(y_eval_enc, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_eval_enc, y_pred, average="macro", zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_eval_enc, y_pred, average="weighted", zero_division=0
    )

    result = {
        "vectorizer": vectorizer_name,
        "model": model_name,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

    return result, y_pred


def show_classification_report(y_true_enc, y_pred_enc, label_encoder):
    print(classification_report(
        y_true_enc,
        y_pred_enc,
        target_names=label_encoder.classes_,
        zero_division=0
    ))


def plot_confusion(y_true_enc, y_pred_enc, label_encoder, title):
    cm = confusion_matrix(y_true_enc, y_pred_enc)

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )
    plt.title(title)
    plt.xlabel("Prédiction")
    plt.ylabel("Vraie classe")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()

## 11. Modèles TF-IDF à comparer

On compare plusieurs pipelines :

1. **TF-IDF + Logistic Regression**
2. **TF-IDF + Linear SVM**
3. **TF-IDF + Random Forest**
4. **TF-IDF + XGBoost**
   
Chaque pipeline contient :
- la vectorisation ;
- le modèle ;
- une petite grille de paramètres.

Le but n’est pas de tester 1000 combinaisons, mais d’avoir une comparaison solide et défendable.

In [ ]:
!pip install xgboost

In [26]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.naive_bayes import MultinomialNB

tfidf_pipelines = {}

tfidf_pipelines["TFIDF_LogisticRegression"] = {
    "pipeline": ImbPipeline([
        ("tfidf", TfidfVectorizer(
            token_pattern=r"\b[a-zA-Z']{2,}\b",
            sublinear_tf=True
        )),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "params": {

    }
}

tfidf_pipelines["TFIDF_NaiveBayes"] = {
    "pipeline": ImbPipeline([
        ("tfidf", TfidfVectorizer(
            token_pattern=r"\b[a-zA-Z']{2,}\b",
            sublinear_tf=True
        )),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", MultinomialNB())
    ]),
    "params": {

    }
}

tfidf_pipelines["TFIDF_LinearSVM"] = {
    "pipeline": ImbPipeline([
        ("tfidf", TfidfVectorizer(
            token_pattern=r"\b[a-zA-Z']{2,}\b",
            sublinear_tf=True
        )),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", LinearSVC(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "params": {

    }
}

tfidf_pipelines["TFIDF_XGBoost"] = {
    "pipeline": ImbPipeline([
        ("tfidf", TfidfVectorizer(
            token_pattern=r"\b[a-zA-Z']{2,}\b",
            sublinear_tf=True
        )),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", XGBClassifier(
            objective="multi:softmax",
            num_class=len(label_encoder.classes_),
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "params": {

    }
}

print("Pipelines disponibles :")
for name in tfidf_pipelines:
    print("-", name)

Pipelines disponibles :
- TFIDF_LogisticRegression
- TFIDF_NaiveBayes
- TFIDF_LinearSVM
- TFIDF_XGBoost


## 12. Entraînement et tuning des modèles TF-IDF

On utilise `GridSearchCV` avec `f1_macro` comme score principal.

Pourquoi `f1_macro` ?
- car le dataset est déséquilibré ;
- car on veut éviter qu’un modèle soit bon uniquement sur les classes majoritaires ;
- car chaque classe doit compter dans l’évaluation.

In [ ]:
tfidf_results = []
trained_tfidf_models = {}
tfidf_predictions_val = {}

for name, config in tfidf_pipelines.items():

    print("\n" + "="*80)
    print("Entraînement :", name)
    print("="*80)

    start_time = time.time()

    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train_enc)

    elapsed = time.time() - start_time

    print("Meilleurs paramètres :")
    print(grid.best_params_)
    print("Meilleur score CV macro F1 :", round(grid.best_score_, 4))
    print("Temps d'entraînement :", round(elapsed, 2), "secondes")

    best_model = grid.best_estimator_
    result, y_pred_val = evaluate_model(
        model_name=name.replace("TFIDF_", ""),
        vectorizer_name="TF-IDF",
        model=best_model,
        X_eval=X_val,
        y_eval_enc=y_val_enc,
        label_encoder=label_encoder
    )

    result["best_cv_f1_macro"] = grid.best_score_
    result["training_time_sec"] = elapsed
    result["best_params"] = str(grid.best_params_)

    tfidf_results.append(result)
    trained_tfidf_models[name] = best_model
    tfidf_predictions_val[name] = y_pred_val

tfidf_results_df = pd.DataFrame(tfidf_results).sort_values(by="f1_macro", ascending=False)
display(tfidf_results_df)


Entraînement : TFIDF_LogisticRegression
Fitting 3 folds for each of 1 candidates, totalling 3 fits


In [ ]:
# =========================
# TABLEAU COMPARATIF DES MODÈLES TF-IDF
# =========================

tfidf_comparison = tfidf_results_df.copy()

cols_to_show = [
    "vectorizer",
    "model",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "best_cv_f1_macro",
    "training_time_sec",
    "best_params"
]

tfidf_comparison = tfidf_comparison[cols_to_show]

tfidf_comparison["training_time_sec"] = tfidf_comparison["training_time_sec"].round(2)
tfidf_comparison["accuracy"] = tfidf_comparison["accuracy"].round(4)
tfidf_comparison["precision_macro"] = tfidf_comparison["precision_macro"].round(4)
tfidf_comparison["recall_macro"] = tfidf_comparison["recall_macro"].round(4)
tfidf_comparison["f1_macro"] = tfidf_comparison["f1_macro"].round(4)
tfidf_comparison["best_cv_f1_macro"] = tfidf_comparison["best_cv_f1_macro"].round(4)

display(tfidf_comparison)

## 13. Analyse du meilleur modèle TF-IDF sur validation

On sélectionne le meilleur modèle TF-IDF selon le **Macro F1** sur le validation set.

Ensuite, on affiche :
- classification report ;
- matrice de confusion ;
- interprétation des classes bien ou mal reconnues.

In [ ]:
best_tfidf_row = tfidf_results_df.iloc[0]
best_tfidf_key = "TFIDF_" + best_tfidf_row["model"]

best_tfidf_model = trained_tfidf_models[best_tfidf_key]
best_tfidf_pred_val = tfidf_predictions_val[best_tfidf_key]

print("Meilleur modèle TF-IDF :", best_tfidf_key)
print("Macro F1 validation :", round(best_tfidf_row["f1_macro"], 4))
print("\nClassification report :")
show_classification_report(y_val_enc, best_tfidf_pred_val, label_encoder)

plot_confusion(
    y_val_enc,
    best_tfidf_pred_val,
    label_encoder,
    title=f"Matrice de confusion validation - {best_tfidf_key}"
)

## 14. Analyse des features importantes du meilleur modèle TF-IDF

Cette partie est possible surtout pour les modèles linéaires :
- Logistic Regression ;
- Linear SVM.

Elle permet de voir quels mots ou expressions influencent chaque classe.

C’est très utile pour l’interprétabilité du projet.

In [ ]:
def show_top_features_linear_model(pipeline, label_encoder, top_n=15):
    vectorizer = pipeline.named_steps["tfidf"]
    clf = pipeline.named_steps["clf"]
    feature_names = np.array(vectorizer.get_feature_names_out())

    if not hasattr(clf, "coef_"):
        print("Ce modèle ne possède pas de coefficients linéaires interprétables.")
        return

    coefs = clf.coef_

    for i, class_name in enumerate(label_encoder.classes_):
        print("\n" + "="*60)
        print("Classe :", class_name)
        print("="*60)

        top_idx = np.argsort(coefs[i])[-top_n:][::-1]
        top_features = pd.DataFrame({
            "feature": feature_names[top_idx],
            "coefficient": coefs[i][top_idx]
        })

        display(top_features)

show_top_features_linear_model(best_tfidf_model, label_encoder, top_n=15)

# Partie B — Word2Vec

## 15. Pourquoi Word2Vec ?

TF-IDF représente les textes par fréquence de mots.  
Word2Vec apprend des vecteurs denses où les mots proches dans le contexte ont des représentations proches.

Dans ce projet, Word2Vec permet de tester une représentation plus sémantique, mais toujours classique et explicable.

## 16. Préparation des tokens pour Word2Vec

Word2Vec travaille sur des listes de tokens.

Important :
- le modèle Word2Vec doit être entraîné uniquement sur le train set ;
- puis on transforme validation et test avec le même espace vectoriel.

In [ ]:
def simple_tokenize(text):
    return re.findall(r"\b[a-zA-Z']{2,}\b", str(text).lower())

train_tokens = [simple_tokenize(text) for text in X_train]
val_tokens = [simple_tokenize(text) for text in X_val]
test_tokens = [simple_tokenize(text) for text in X_test]

print("Exemple de tokens :")
print(train_tokens[0][:30])

token_lengths = [len(tokens) for tokens in train_tokens]
print("\nLongueur moyenne en tokens train :", round(np.mean(token_lengths), 2))

## 17. Entraînement Word2Vec

On entraîne Word2Vec sur le train set.

Paramètres importants :
- `vector_size` : dimension du vecteur ;
- `window` : taille du contexte ;
- `min_count` : fréquence minimale d’un mot ;
- `sg=1` : Skip-gram ;
- `sg=0` : CBOW.

Pour commencer, on teste une configuration raisonnable.

In [ ]:
try:
    from gensim.models import Word2Vec

    w2v_model = Word2Vec(
        sentences=train_tokens,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4,
        sg=1,
        epochs=10,
        seed=RANDOM_STATE
    )

    print("Word2Vec entraîné avec succès.")
    print("Taille du vocabulaire Word2Vec :", len(w2v_model.wv.index_to_key))

except Exception as e:
    w2v_model = None
    print("Impossible d'entraîner Word2Vec.")
    print("Installez gensim avec : pip install gensim")
    print("Erreur :", e)

## 18. Transformation des textes en vecteurs Word2Vec

Pour représenter un texte complet, on calcule la moyenne des vecteurs de ses mots.

C’est une méthode simple, mais très utile comme baseline Word2Vec.

In [ ]:
def document_vector(tokens, model, vector_size=100):
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)


if w2v_model is not None:
    X_train_w2v = np.vstack([document_vector(tokens, w2v_model, 100) for tokens in train_tokens])
    X_val_w2v = np.vstack([document_vector(tokens, w2v_model, 100) for tokens in val_tokens])
    X_test_w2v = np.vstack([document_vector(tokens, w2v_model, 100) for tokens in test_tokens])

    print("Shape Word2Vec train :", X_train_w2v.shape)
    print("Shape Word2Vec validation :", X_val_w2v.shape)
    print("Shape Word2Vec test :", X_test_w2v.shape)
else:
    print("Word2Vec non disponible.")

In [ ]:
# oversampling using RandomOverSampler

In [ ]:
# Oversampling using RandomOverSampler
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train_w2v, y_train = ros.fit_resample(
    np.vstack([document_vector(tokens, w2v_model, 100) for tokens in train_tokens]),
    y_train
)
X_val_w2v = np.vstack([document_vector(tokens, w2v_model, 100) for tokens in val_tokens])
X_test_w2v = np.vstack([document_vector(tokens, w2v_model, 100) for tokens in test_tokens])

## 19. Modèles sur Word2Vec

On compare les mêmes familles de modèles :

- Logistic Regression ;
- Linear SVM ;
- Random Forest ;
- XGBoost si disponible.

Ici les vecteurs sont denses, donc Random Forest et XGBoost peuvent parfois mieux se comporter que sur TF-IDF sparse.

In [ ]:
w2v_models = {}

if w2v_model is not None:

    w2v_models["W2V_LogisticRegression"] = {
        "model": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "C": [0.5, 1.0, 2.0]
        }
    }

    w2v_models["W2V_LinearSVM"] = {
        "model": LinearSVC(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "params": {
            "C": [0.5, 1.0, 2.0]
        }
    }

    w2v_models["W2V_RandomForest"] = {
        "model": RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [200, 300],
            "max_depth": [None, 30]
        }
    }

    try:
        from xgboost import XGBClassifier

        w2v_models["W2V_XGBoost"] = {
            "model": XGBClassifier(
                objective="multi:softmax",
                num_class=len(label_encoder.classes_),
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),
            "params": {
                "n_estimators": [200],
                "max_depth": [4, 6],
                "learning_rate": [0.05, 0.1]
            }
        }

    except Exception:
        pass

    print("Modèles Word2Vec disponibles :")
    for name in w2v_models:
        print("-", name)
else:
    print("Aucun modèle Word2Vec disponible.")

## 20. Entraînement des modèles Word2Vec

On utilise encore `GridSearchCV` avec `f1_macro`.

Comme les matrices Word2Vec sont denses et plus petites, l’entraînement est souvent plus simple.

In [ ]:
w2v_results = []
trained_w2v_models = {}
w2v_predictions_val = {}

if w2v_model is not None:

    for name, config in w2v_models.items():

        print("\n" + "="*80)
        print("Entraînement :", name)
        print("="*80)

        start_time = time.time()

        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            scoring="f1_macro",
            cv=3,
            n_jobs=-1,
            verbose=1
        )

        grid.fit(X_train_w2v, y_train_enc)

        elapsed = time.time() - start_time

        print("Meilleurs paramètres :")
        print(grid.best_params_)
        print("Meilleur score CV macro F1 :", round(grid.best_score_, 4))
        print("Temps d'entraînement :", round(elapsed, 2), "secondes")

        best_model = grid.best_estimator_

        result, y_pred_val = evaluate_model(
            model_name=name.replace("W2V_", ""),
            vectorizer_name="Word2Vec",
            model=best_model,
            X_eval=X_val_w2v,
            y_eval_enc=y_val_enc,
            label_encoder=label_encoder
        )

        result["best_cv_f1_macro"] = grid.best_score_
        result["training_time_sec"] = elapsed
        result["best_params"] = str(grid.best_params_)

        w2v_results.append(result)
        trained_w2v_models[name] = best_model
        w2v_predictions_val[name] = y_pred_val

    w2v_results_df = pd.DataFrame(w2v_results).sort_values(by="f1_macro", ascending=False)
    display(w2v_results_df)

else:
    w2v_results_df = pd.DataFrame()
    print("Word2Vec n'a pas été exécuté.")

# Partie C — Comparaison globale

## 21. Comparaison TF-IDF vs Word2Vec

On regroupe tous les résultats validation dans un seul tableau.

Le meilleur modèle sera choisi principalement selon :
- Macro F1 ;
- Recall macro ;
- stabilité ;
- interprétabilité ;
- temps d’entraînement.

In [ ]:
all_results_df = pd.concat(
    [tfidf_results_df, w2v_results_df],
    ignore_index=True
).sort_values(by="f1_macro", ascending=False)

display(all_results_df)

plt.figure(figsize=(12, 6))
sns.barplot(data=all_results_df, x="f1_macro", y="model", hue="vectorizer")
plt.title("Comparaison des modèles selon Macro F1 sur validation")
plt.xlabel("Macro F1")
plt.ylabel("Modèle")
plt.show()

### Interprétation attendue

Si TF-IDF + Linear SVM ou TF-IDF + Logistic Regression arrive en tête, c’est un résultat très cohérent en classification de texte classique.

Si Word2Vec est moins performant, ce n’est pas forcément un échec :
- TF-IDF capture directement les mots discriminants ;
- Word2Vec moyenne les vecteurs et peut perdre certains signaux précis ;
- les bigrams TF-IDF peuvent mieux capturer des expressions comme `not okay`, `kill myself`, `feel empty`.

## 22. Sélection automatique du meilleur modèle validation

On choisit le modèle avec le meilleur Macro F1 validation.

Attention : le test set n’est pas encore utilisé pour choisir le modèle.  
Il sert seulement à l’évaluation finale.

In [ ]:
best_row = all_results_df.iloc[0]

print("Meilleur modèle sur validation :")
display(best_row.to_frame().T)

best_vectorizer = best_row["vectorizer"]
best_model_name = best_row["model"]

if best_vectorizer == "TF-IDF":
    best_key = "TFIDF_" + best_model_name
    best_model = trained_tfidf_models[best_key]
    best_X_test = X_test
    best_X_val = X_val
    best_pred_val = tfidf_predictions_val[best_key]

elif best_vectorizer == "Word2Vec":
    best_key = "W2V_" + best_model_name
    best_model = trained_w2v_models[best_key]
    best_X_test = X_test_w2v
    best_X_val = X_val_w2v
    best_pred_val = w2v_predictions_val[best_key]

print("Clé du meilleur modèle :", best_key)

## 23. Évaluation finale sur le test set

Maintenant seulement, on évalue le meilleur modèle sur le test set.

Cette étape donne la performance finale réaliste.

In [ ]:
test_result, y_pred_test = evaluate_model(
    model_name=best_model_name,
    vectorizer_name=best_vectorizer,
    model=best_model,
    X_eval=best_X_test,
    y_eval_enc=y_test_enc,
    label_encoder=label_encoder
)

print("Résultats finaux sur test set :")
display(pd.DataFrame([test_result]))

print("\nClassification report test :")
show_classification_report(y_test_enc, y_pred_test, label_encoder)

plot_confusion(
    y_test_enc,
    y_pred_test,
    label_encoder,
    title=f"Matrice de confusion test - {best_vectorizer} + {best_model_name}"
)

## 24. Analyse des erreurs

On affiche quelques exemples mal classés.

Cette analyse est très importante pour comprendre :
- les classes confondues ;
- les textes ambigus ;
- les limites du modèle ;
- les prochaines améliorations possibles.

In [ ]:
error_df = pd.DataFrame({
    "text": X_test,
    "true_label": label_encoder.inverse_transform(y_test_enc),
    "predicted_label": label_encoder.inverse_transform(y_pred_test)
})

error_df = error_df[error_df["true_label"] != error_df["predicted_label"]].copy()

print("Nombre d'erreurs sur le test set :", len(error_df))
display(error_df.head(20))

## 25. Analyse des confusions principales

On identifie les paires de classes les plus souvent confondues.

Exemple possible :
- Depression confondue avec Suicidal ;
- Anxiety confondue avec Stress ;
- Bipolar confondu avec Depression.

Cela aide à expliquer les limites du modèle dans le rapport.

In [ ]:
confusion_pairs = error_df.groupby(
    ["true_label", "predicted_label"]
).size().reset_index(name="count").sort_values(by="count", ascending=False)

display(confusion_pairs.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=confusion_pairs.head(15),
    x="count",
    y=confusion_pairs.head(15).apply(lambda row: f"{row['true_label']} → {row['predicted_label']}", axis=1)
)
plt.title("Top confusions entre classes")
plt.xlabel("Nombre d'erreurs")
plt.ylabel("Confusion")
plt.show()

## 26. Sauvegarde du meilleur modèle

On sauvegarde :
- le meilleur modèle ;
- le label encoder ;
- le tableau des résultats ;
- les prédictions d’erreur.

Cela permet de réutiliser le modèle plus tard dans un dashboard ou une application.

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

joblib.dump(best_model, "models/best_model_step03.joblib")
joblib.dump(label_encoder, "models/label_encoder_step03.joblib")

all_results_df.to_csv("outputs/model_comparison_step03.csv", index=False)
error_df.to_csv("outputs/test_errors_step03.csv", index=False)

print("Modèle sauvegardé dans : models/best_model_step03.joblib")
print("Label encoder sauvegardé dans : models/label_encoder_step03.joblib")
print("Résultats sauvegardés dans : outputs/model_comparison_step03.csv")
print("Erreurs sauvegardées dans : outputs/test_errors_step03.csv")

if best_vectorizer == "Word2Vec" and w2v_model is not None:
    w2v_model.save("models/word2vec_model_step03.model")
    print("Word2Vec sauvegardé dans : models/word2vec_model_step03.model")

## 27. Synthèse automatique de l’étape 03

Cette synthèse peut être réutilisée dans le rapport.

In [ ]:
summary = []

summary.append("Le dataset nettoyé issu du Step 02 a été utilisé pour la modélisation.")
summary.append("Un split stratifié Train / Validation / Test a été appliqué afin de conserver la distribution des classes.")
summary.append("Les labels textuels ont été encodés numériquement avec LabelEncoder.")
summary.append("Le déséquilibre des classes a été pris en compte avec des poids de classes pour les modèles compatibles.")
summary.append("La représentation TF-IDF a été testée avec plusieurs configurations de n-grams et de taille de vocabulaire.")
summary.append("Des modèles classiques ont été comparés : Logistic Regression, Linear SVM, Random Forest et XGBoost si disponible.")
summary.append("Word2Vec a également été testé comme représentation dense et sémantique.")
summary.append(f"Le meilleur modèle sur validation est : {best_vectorizer} + {best_model_name}.")
summary.append(f"Son Macro F1 sur validation est : {round(best_row['f1_macro'], 4)}.")
summary.append(f"Son Macro F1 final sur test est : {round(test_result['f1_macro'], 4)}.")
summary.append("Les erreurs de classification ont été analysées pour identifier les classes les plus confondues.")

print("SYNTHÈSE STEP 03")
print("="*60)

for item in summary:
    print("- " + item)

# Conclusion de l’étape 03

r :

- une comparaison claire entre TF-IDF et Word2Vec ;
- plusieurs modèles classiques entraînés ;
- une sélection justifiée du meilleur modèle ;
- une évaluation finale sur test set ;
- une analyse des erreurs ;
- un modèle sauvegardé pour la suite.

La prochaine étape logique sera :

```text
Step 04 — Model Optimization & Deployment Preparation
```

Mais avant cela, il faut analyser calmement les résultats obtenus dans ce notebook.